# 9주차 _ 1팀

# 자료불러오기

In [54]:
# 1) 내 Google Drive 연결 — 원시자료는 각자 자기 Drive 에 둡니다
from google.colab import drive
drive.mount('/content/drive')
import pyreadstat
DATA_DIR = '/content/drive/MyDrive/hdac_data' # 팀 공통 경로

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [55]:
# ==============================================================================
# Google Drive 연결 및 SAS 원시자료 로드 (실제 변수명 반영)
# ==============================================================================
from google.colab import drive
drive.mount('/content/drive')

!pip -q install pyreadstat
import pyreadstat
import pandas as pd
import numpy as np

# 팀 공통 경로 및 파일명 설정
DATA_DIR = '/content/drive/MyDrive/원시자료'
FILE_NAME = 'kyrbs2025.sas7bdat'

# 분석에 사용할 실제 원변수명 목록
cols = [
    'M_SLP_HR_K',   # 주중 잠든 시각 (시간) - 노출 산출용
    'M_SLP_MM_K',   # 주중 잠든 시각 (분) - 노출 산출용
    'M_WK_HR_K',    # 주중 일어난 시각 (시간) - 노출 산출용
    'M_WK_MM_K',    # 주중 일어난 시각 (분) - 노출 산출용
    'E_S_RCRD',     # 주관적 학업성적 (결과)
    'SEX',          # 성별 (공변량)
    'GRADE',        # 학년 (공변량)
    'E_SES',        # 가구 경제 상태 (공변량)
    'M_STR',        # 평상시 스트레스 인지 (공변량)
    'PA_SWD_S',     # 주중 학습 목적 앉아서 보낸 시간 (공변량)
    'F_BR'          # 아침식사 횟수 등 (공변량)
]

print("SAS 원시자료를 불러오는 중입니다...")

# 파일 경로 조합 및 데이터 로드
file_path = f'{DATA_DIR}/{FILE_NAME}'
df, meta = pyreadstat.read_sas7bdat(file_path, usecols=cols)

print(f"[로드 완료] 전체 데이터 형태(행, 열): {df.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SAS 원시자료를 불러오는 중입니다...
[로드 완료] 전체 데이터 형태(행, 열): (54170, 11)


# 환경설정(분석용 변수 구성)

In [56]:
from IPython.display import display

pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 160)

# =========================================================
# 0. 분석용 변수 구성 (기본설정의 cols, df 사용)
# =========================================================
EXPO_VARS = ['M_SLP_HR_K', 'M_SLP_MM_K', 'M_WK_HR_K', 'M_WK_MM_K']
OUTCOME   = 'E_S_RCRD'
COVARS    = ['SEX', 'GRADE', 'E_SES', 'M_STR', 'PA_SWD_S', 'F_BR']
assert set(EXPO_VARS + [OUTCOME] + COVARS) == set(cols), "cols와 변수 구성이 다릅니다."

# 유효값 범위: 반드시 이용지침서(코드북)와 대조해 확정할 것
VALID = {
    'M_SLP_HR_K': range(0, 24),
    'M_SLP_MM_K': range(0, 60),
    'M_WK_HR_K':  range(0, 24),
    'M_WK_MM_K':  range(0, 60),
    'E_S_RCRD':   range(1, 6),   # 1=상 ... 5=하
    'SEX':        [1, 2],
    'GRADE':      range(1, 7),   # 1~3 중1~3, 4~6 고1~3
    'E_SES':      range(1, 6),
    'M_STR':      range(1, 6),
    'PA_SWD_S':   range(1, 9),   # 확인 필요
    'F_BR':       range(1, 9),   # 확인 필요
}

# 설계 변수 후보: 이용지침서의 변수명으로 확정해 수정
WEIGHT_CANDS  = ['W']
STRATA_CANDS  = ['STRATA']
CLUSTER_CANDS = ['CLUSTER']

In [57]:
# =========================================================
# 1. 설계 변수(가중치·층·집락)만 추가로 읽어 붙이기
# =========================================================
_, meta_all = pyreadstat.read_sas7bdat(file_path, metadataonly=True)
all_cols = list(meta_all.column_names)

def find_var(cands):
    return next((c for c in cands if c in all_cols), None)

W_VAR, STRATA_VAR, CLUSTER_VAR = map(find_var, [WEIGHT_CANDS, STRATA_CANDS, CLUSTER_CANDS])
design_vars = [v for v in [W_VAR, STRATA_VAR, CLUSTER_VAR] if v]

if design_vars:
    df_design, _ = pyreadstat.read_sas7bdat(file_path, usecols=design_vars)
    assert len(df_design) == len(df), "행 수가 달라 설계 변수를 붙일 수 없습니다."
    df_raw = pd.concat([df.reset_index(drop=True), df_design.reset_index(drop=True)], axis=1)
else:
    df_raw = df.copy()

# 2. 변수확인

In [58]:
print("=" * 60 + "\n[2] 변수 확인\n" + "=" * 60)
print("원시자료 행 수:", len(df_raw), "/ 읽은 변수 수:", df_raw.shape[1])
print(f"가중치={W_VAR}, 층={STRATA_VAR}, 집락={CLUSTER_VAR}")
if not (W_VAR and STRATA_VAR and CLUSTER_VAR):
    hint = [c for c in all_cols
            if any(k in c.upper() for k in ["W", "STRA", "CLUS", "PSU"]) and len(c) <= 12]
    print("※ 설계 변수 일부를 못 찾음. 이름에 W/STRA/CLUS/PSU가 들어간 열:", hint)

summary = pd.DataFrame({
    "dtype":    df_raw[cols].dtypes.astype(str),
    "n_nan":    df_raw[cols].isna().sum(),
    "n_unique": df_raw[cols].nunique(),
    "min":      df_raw[cols].min(numeric_only=True),
    "max":      df_raw[cols].max(numeric_only=True),
})
display(summary)
non_num = [c for c in cols if not pd.api.types.is_numeric_dtype(df_raw[c])]
print("숫자형 아닌 변수:", non_num if non_num else "없음")

[2] 변수 확인
원시자료 행 수: 54170 / 읽은 변수 수: 14
가중치=W, 층=STRATA, 집락=CLUSTER


,dtype,n_nan,n_unique,min,max
M_SLP_HR_K,float64,0,24,1.0,24.0
M_SLP_MM_K,float64,0,6,0.0,50.0
M_WK_HR_K,float64,0,24,1.0,24.0
M_WK_MM_K,float64,0,6,0.0,50.0
E_S_RCRD,float64,6,5,1.0,5.0
SEX,float64,0,2,1.0,2.0
GRADE,float64,0,6,1.0,6.0
E_SES,float64,12,5,1.0,5.0
M_STR,float64,0,5,1.0,5.0
PA_SWD_S,float64,703,598,0.0,1438.0


숫자형 아닌 변수: 없음


# 3. 원 코드 빈도표

In [59]:
print("\n" + "=" * 60 + "\n[3] 원 코드 빈도표\n" + "=" * 60)

def freq_table(s):
    vc = s.value_counts(dropna=False).sort_index()
    out = pd.DataFrame({"n": vc, "%": (vc / vc.sum() * 100).round(2)})
    out.index.name = s.name
    return out

for v in cols:
    print(f"\n----- {v} -----")
    display(freq_table(df_raw[v]))


[3] 원 코드 빈도표

----- M_SLP_HR_K -----


,n,%
M_SLP_HR_K,,
1.0,10416,19.23
2.0,11687,21.57
3.0,8131,15.01
4.0,3084,5.69
5.0,1242,2.29
6.0,434,0.80
7.0,230,0.42
8.0,166,0.31
9.0,133,0.25



----- M_SLP_MM_K -----


,n,%
M_SLP_MM_K,,
0.0,31644,58.42
10.0,3818,7.05
20.0,3218,5.94
30.0,10888,20.10
40.0,2643,4.88
50.0,1959,3.62



----- M_WK_HR_K -----


...

,n,%
M_STR,,
1.0,6034,11.14
2.0,16221,29.94
3.0,23059,42.57
4.0,7220,13.33
5.0,1636,3.02



----- PA_SWD_S -----


,n,%
PA_SWD_S,,
0.0,1683,3.11
1.0,57,0.11
2.0,5,0.01
3.0,6,0.01
5.0,23,0.04
...,...,...
1409.0,1,0.00
1410.0,1,0.00
1430.0,1,0.00



----- F_BR -----


,n,%
F_BR,,
1.0,14308,26.41
2.0,4236,7.82
3.0,5105,9.42
4.0,4153,7.67
5.0,3266,6.03
6.0,5859,10.82
7.0,3306,6.10
8.0,13937,25.73


# 4. 무응답·비해당 코드 결측 처리 및 수면시간 산출

In [60]:
d = df_raw.copy()
for v, rng in VALID.items():
    d.loc[~d[v].isin(list(rng)), v] = np.nan

bed  = d['M_SLP_HR_K'] * 60 + d['M_SLP_MM_K']
wake = d['M_WK_HR_K']  * 60 + d['M_WK_MM_K']
d['SLEEP_HR'] = ((wake - bed) % (24 * 60)) / 60   # 자정 넘김 보정

d['EXPOSURE'] = np.nan
d.loc[d['SLEEP_HR'] < 8, 'EXPOSURE'] = 1                  # 수면부족(<8h)
d.loc[d['SLEEP_HR'].between(8, 10), 'EXPOSURE'] = 0       # 권장(8~10h)
d.loc[d['SLEEP_HR'] > 10, 'EXPOSURE'] = 2                 # 참고용(>10h)

print("\n" + "=" * 60 + "\n[4] 수면시간 산출 결과\n" + "=" * 60)
display(d['SLEEP_HR'].describe())
print("노출군 분포 (0=8~10h, 1=<8h, 2=>10h, NaN=산출불가)")
display(freq_table(d['EXPOSURE']))


[4] 수면시간 산출 결과


,SLEEP_HR
count,45470.000000
mean,8.780694
std,3.741614
min,0.000000
25%,7.000000
50%,8.000000
75%,9.500000
max,23.833333


노출군 분포 (0=8~10h, 1=<8h, 2=>10h, NaN=산출불가)


,n,%
EXPOSURE,,
0.0,18336,33.85
1.0,19179,35.41
2.0,7955,14.69
NaN,8700,16.06


# 5. 대상 선정 단계별 인원

In [61]:
print("\n" + "=" * 60 + "\n[5] 대상 선정 단계별 인원\n" + "=" * 60)
steps = []
def log(label, criterion, x):
    n = len(x)
    prev = steps[-1]["남은 인원(명)"] if steps else n
    steps.append({
        "단계": label, "기준": criterion,
        "남은 인원(명)": n, "단계 제외(명)": prev - n,
        "누적 제외율(%)": round((1 - n / len(d)) * 100, 2),
        "가중합": round(x[W_VAR].sum()) if W_VAR else None,
    })

s1 = d;                                    log("1단계", "전체 원시자료 응답자", s1)
s2 = s1[s1['GRADE'].between(1, 6)];        log("2단계", "중1~고3 재학생(GRADE 1~6)", s2)
s3 = s2.dropna(subset=['SLEEP_HR']);       log("3단계", "노출(주중 수면시간) 결측 제외", s3)
s4 = s3.dropna(subset=[OUTCOME]);          log("4단계", "결과(주관적 학업성적) 결측 제외", s4)
s5 = s4.dropna(subset=COVARS);             log("5단계", "보정변수 결측 제외(최종)", s5)

display(pd.DataFrame(steps))

print("\n4단계 대상자 중 보정변수별 결측 수")
display(s4[COVARS].isna().sum().rename("n_missing").to_frame())

df_final = s5.copy()
print("\n최종 분석 대상 노출군 분포 (0=8~10h, 1=<8h, 2=>10h)")
display(freq_table(df_final['EXPOSURE']))
print("최종 분석 대상 N =", len(df_final))

# 결과 문서에는 이 출력의 숫자를 그대로 복사해 쓸 것
print("\n[보고서용 표]")
print("| 단계 | 기준 | 남은 인원(명) | 단계 제외(명) |")
print("|---|---|---|---|")
for r in steps:
    print(f"| {r['단계']} | {r['기준']} | {r['남은 인원(명)']:,} | {r['단계 제외(명)']:,} |")



[5] 대상 선정 단계별 인원


,단계,기준,남은 인원(명),단계 제외(명),누적 제외율(%),가중합
0,1단계,전체 원시자료 응답자,54170,0,0.00,2642704
1,2단계,중1~고3 재학생(GRADE 1~6),54170,0,0.00,2642704
2,3단계,노출(주중 수면시간) 결측 제외,45470,8700,16.06,2228385
3,4단계,결과(주관적 학업성적) 결측 제외,45466,4,16.07,2228182
4,5단계,보정변수 결측 제외(최종),84,45382,99.84,4067



4단계 대상자 중 보정변수별 결측 수


,n_missing
SEX,0
GRADE,0
E_SES,6
M_STR,0
PA_SWD_S,45382
F_BR,0



최종 분석 대상 노출군 분포 (0=8~10h, 1=<8h, 2=>10h)


,n,%
EXPOSURE,,
0.0,32,38.10
1.0,23,27.38
2.0,29,34.52


최종 분석 대상 N = 84

[보고서용 표]
| 단계 | 기준 | 남은 인원(명) | 단계 제외(명) |
|---|---|---|---|
| 1단계 | 전체 원시자료 응답자 | 54,170 | 0 |
| 2단계 | 중1~고3 재학생(GRADE 1~6) | 54,170 | 0 |
| 3단계 | 노출(주중 수면시간) 결측 제외 | 45,470 | 8,700 |
| 4단계 | 결과(주관적 학업성적) 결측 제외 | 45,466 | 4 |
| 5단계 | 보정변수 결측 제외(최종) | 84 | 45,382 |


#6. 가중치·층·집락 확인

In [62]:
print("\n" + "=" * 60 + "\n[6] 가중치·층·집락 확인\n" + "=" * 60)

def design_check(x, name):
    print(f"\n----- {name} (n={len(x):,}) -----")
    if not design_vars:
        print("설계 변수를 찾지 못해 건너뜀"); return
    print("설계변수 결측:", x[design_vars].isna().sum().to_dict())
    if W_VAR:
        display(x[W_VAR].describe())
        print("가중치 0 이하:", int((x[W_VAR] <= 0).sum()))
        print("가중합(추정 모집단):", f"{x[W_VAR].sum():,.0f}")
        w = x[W_VAR]
        print("Kish 설계효과(가중치 변동만):", round(len(w) * (w**2).sum() / w.sum()**2, 3))
    if STRATA_VAR:
        print("층 수:", x[STRATA_VAR].nunique())
    if CLUSTER_VAR:
        print("집락 수:", x[CLUSTER_VAR].nunique())
    if STRATA_VAR and CLUSTER_VAR:
        psu = x.groupby(STRATA_VAR)[CLUSTER_VAR].nunique()
        display(psu.describe())
        print("집락이 1개뿐인 층(lonely PSU) 수:", int((psu < 2).sum()))

design_check(df_raw, "전체 원시자료")
design_check(df_final, "최종 분석 대상")

if STRATA_VAR and CLUSTER_VAR:
    comp = pd.DataFrame({
        "전체": [df_raw[STRATA_VAR].nunique(), df_raw[CLUSTER_VAR].nunique()],
        "최종": [df_final[STRATA_VAR].nunique(), df_final[CLUSTER_VAR].nunique()],
    }, index=["층 수", "집락 수"])
    print("\n선정 전후 층·집락 수 비교")
    display(comp)


[6] 가중치·층·집락 확인

----- 전체 원시자료 (n=54,170) -----
설계변수 결측: {'W': 0, 'STRATA': 0, 'CLUSTER': 0}


,W
count,54170.000000
mean,48.785379
std,22.444107
min,3.659465
25%,32.779359
50%,46.370047
75%,61.761557
max,298.000000


가중치 0 이하: 0
가중합(추정 모집단): 2,642,704
Kish 설계효과(가중치 변동만): 1.212
층 수: 106
집락 수: 796


,CLUSTER
count,106.000000
mean,7.509434
std,4.799296
min,2.000000
25%,3.000000
50%,7.000000
75%,10.750000
max,24.000000


집락이 1개뿐인 층(lonely PSU) 수: 0

----- 최종 분석 대상 (n=84) -----
설계변수 결측: {'W': 0, 'STRATA': 0, 'CLUSTER': 0}


,W
count,84.000000
mean,48.419231
std,27.660109
min,8.041197
25%,29.495503
50%,44.540088
75%,60.072141
max,187.555726


가중치 0 이하: 0
가중합(추정 모집단): 4,067
Kish 설계효과(가중치 변동만): 1.322
층 수: 55
집락 수: 78


,CLUSTER
count,55.000000
mean,1.418182
std,0.599101
min,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,3.000000


집락이 1개뿐인 층(lonely PSU) 수: 35

선정 전후 층·집락 수 비교


,전체,최종
층 수,106,55
집락 수,796,78
